<a href="https://colab.research.google.com/github/JoGabTasca/NLP/blob/main/Analise_sentimento_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Análise de Sentimentos - Olist dataset

Este notebook replica o exemplo fornecido para realizar análise de sentimentos em comentários de usuários do dataset Brazilian E-Commerce Public Dataset by Olist. O objetivo é classificar as avaliações como positivas ou negativas.

### Pré-processamento

Importando as bibliotecas necessárias.

In [ ]:
import pandas as pd
import re
import nltk
import spacy
import matplotlib.pyplot as plt
import seaborn as sns

Carregando e organizando o dataset.

In [ ]:
# Carregando o dataset
data = pd.read_csv("https://github.com/cassiusf/datasets/raw/refs/heads/main/nlp/olist_order_reviews_dataset.csv")

In [ ]:
print('Primeiras 5 linhas do dataset:')
display(data.head())

Como nosso foco é a análise de sentimentos, não precisamos das colunas `order_id`, `review_creation_date` e `review_answer_timestamp`. Além disso, vamos verificar a estrutura do dataframe.

In [ ]:
data.drop(['order_id', 'review_creation_date', 'review_answer_timestamp'], axis=1, inplace=True)
print('Informações do dataframe após remoção de colunas:')
data.info()

Temos pouco menos de 100.000 linhas, todas possuindo um escore de avaliação, mas nem todos possuem análise escrita. Vamos então verificar se há dados duplicados (com base no id) e removê-los.

In [ ]:
duplicate_id_percentage = (data.duplicated(subset=['review_id']).sum() / len(data)) * 100
print(f"Reviews com id duplicado: {duplicate_id_percentage:.2f}%")
data.drop_duplicates("review_id", inplace=True)
print(f"Número de linhas após remover duplicados: {len(data)}")

Como estamos interessados justamente no texto, vamos juntar as colunas `review_comment_title` e `review_comment_message` em uma só e tirar entradas que não possuem texto.

In [ ]:
# Para não ter problemas com nulos na concatenação
data.fillna('', inplace=True)

# Concatenando as duas colunas
data['review'] = data['review_comment_title'] + ' ' + data['review_comment_message']

# Removendo entradas sem texto
data = data[data['review'] != ' ']
print('Informações do dataframe após concatenar reviews e remover vazios:')
data.info()
print('Primeiras 5 linhas do dataset com a nova coluna review:')
display(data.head())

### Review scores

Vamos verificar a distribuição das notas e transformar o problema em binário.

In [ ]:
print('Contagem de cada review_score:')
print(data['review_score'].value_counts())

# Se nota for menor ou igual a 3, consideraremos negativa (0) e caso contrário, positiva (1).
labels = []

for score in data['review_score']:
  if score > 3:
    labels.append(1)
  else:
    labels.append(0)

data['label'] = labels
print('\nPrimeiras 10 linhas com a nova coluna label:')
display(data.head(10))

Após este ajuste, vamos verificar o balanceamento final de nossa base.

In [ ]:
plt.figure(figsize=(8,6))
ax = sns.countplot(x='label', data=data, stat="percent")

for p in ax.patches:
    percentage = '{:.1f}%'.format(p.get_height())
    x = p.get_x() + p.get_width() / 2
    y = p.get_height()
    ax.annotate(percentage, (x, y), ha='center', va='bottom')
plt.title('Distribuição de Labels (0: Negativo, 1: Positivo)')
plt.xlabel('Label de Sentimento')
plt.ylabel('Porcentagem')
plt.show()

### Pré-processamento do texto

Normalização para minúsculas; Filtrar apenas letras (removendo pontuações, símbolos, etc.); Remover stopwords; Lematizar.

Para remoção de stopwords, vamos usar a lista disponível na biblioteca `nltk`.

In [ ]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stopwords_pt = stopwords.words("portuguese")
print('Exemplo de stopwords em português:')
print(stopwords_pt[:10])

Nesta lista constam, por exemplo, as palavras `não` e `nem` que, se removidas do texto, podem mudar a interpretação do sentimento de uma frase. Vamos ajustar isso.

In [ ]:
if 'não' in stopwords_pt:
    stopwords_pt.remove('não')
if 'nem' in stopwords_pt:
    stopwords_pt.remove('nem')
print('Stopwords após remoção de "não" e "nem":')
print(stopwords_pt[:10])

Vamos utilizar o lematizador da biblioteca Spacy. Para isso, precisamos fazer o download do modelo de português correspondente.

In [ ]:
import spacy.cli
# Apenas execute o download se o modelo não estiver presente
try:
    spacy.load("pt_core_news_sm")
    print("Modelo 'pt_core_news_sm' já está baixado.")
except OSError:
    print("Baixando modelo 'pt_core_news_sm'...")
    spacy.cli.download("pt_core_news_sm")
    print("Download e instalação bem-sucedidos.")

Após o download, podemos importar e carregar o modelo.

In [ ]:
import pt_core_news_sm
spc_pt = pt_core_news_sm.load()
print('Modelo SpaCy de português carregado com sucesso.')

Agora estamos prontos para criar uma função que realiza os pré-processamentos e aplicá-la no dataset!

A função recebe uma string, deixa tudo em minúsculo, filtra apenas letras, retira stopwords, lemmatiza e retorna a string resultante.

In [ ]:
def limpa_texto(texto):
  texto = texto.lower()

  texto = re.sub(r"[\W\d_]+", " ", texto)

  texto = [pal for pal in texto.split() if pal not in stopwords_pt]

  spc_texto = spc_pt(" ".join(texto))
  tokens = [word.lemma_ if word.lemma_ != "-PRON-" else word.lower_ for word in spc_texto]

  return " ".join(tokens)

In [ ]:
print('Aplicando a função limpa_texto na coluna review (isso pode levar alguns minutos)...')
# Demora aprox. 4 minutos!
data['review'] = data['review'].apply(limpa_texto)
print('Processamento de texto concluído.')
print('Informações do dataframe após limpeza de texto:')
data.info()
print('Primeiras 10 linhas do dataset após limpeza de texto:')
display(data.head(10))

Como temos análises com apenas números ou símbolos, ainda pode haver dados faltantes na coluna `review` após a limpeza, vamos removê-los.

In [ ]:
data = data[data['review'] != '']
print('Informações do dataframe após remover reviews vazias:')
data.info()

### Feature extraction

Essa etapa consiste em transformar o texto em informação numérica, mais precisamente um vetor, para que seja possível utilizá-lo para alimentar um modelo.

Vamos usar a biblioteca Scikit-Learn para realizar as duas técnicas de feature extraction que já vimos em aula, Bag of Words e TF-IDF.

#### Bag of Words

A função `CountVectorizer` do Scikit-Learn foi criada para fazer isso! O parâmetro `binary` indica se será usada a contagem total das palavras ou apenas 0s e 1s. Já o parâmetro `max_features` define o tamanho do vocabulário, assim `max_features=5000` constrói o vocabulário com as 5.000 palavras que mais ocorrem nos textos.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Instanciando o CountVectorizer, binary=True faz a codificação binária
vectorizer = CountVectorizer(binary=True, max_features=5000)

texto = data['review']

# Vetorizando o texto
X_bow = vectorizer.fit_transform(texto)
print('Shape do X_bow (Bag of Words):', X_bow.shape)

#### TF-IDF

Já essa técnica vetoriza o texto atribuindo para cada palavra uma pontuação que mede a importância dela no texto. Também temos uma função no Scikit-Learn que já faz isso, a `TfidfVectorizer`.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Instanciando o TfidfVectorizer
tfidf_vect = TfidfVectorizer(max_features=5000)

# Vetorizando
X_tfidf = tfidf_vect.fit_transform(texto)
print('Shape do X_tfidf (TF-IDF):', X_tfidf.shape)

### Modelos

Dividindo os dados em base de treino e teste (70-30).

In [ ]:
from sklearn.model_selection import train_test_split

# Texto processado com BoW
X1_train, X1_test, y1_train, y1_test = train_test_split(X_bow, data['label'], test_size=0.3, random_state = 10)

# Texto processado com TF-IDF
X2_train, X2_test, y2_train, y2_test = train_test_split(X_tfidf, data['label'], test_size=0.3, random_state = 10)

print(f"Tamanho do conjunto de treino (BoW): {X1_train.shape[0]} amostras")
print(f"Tamanho do conjunto de teste (BoW): {X1_test.shape[0]} amostras")

Importando as métricas que serão usadas para avaliação de cada modelo.

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_auc_score, ConfusionMatrixDisplay

def mostra_metricas(y_true, y_pred):
  print("Acurácia: ", accuracy_score(y_true, y_pred))
  print("\nF1-Score:", f1_score(y_true, y_pred, average='weighted'))
  print("\nMatriz de confusão:")
  cm = confusion_matrix(y_true, y_pred)
  cm_disp = ConfusionMatrixDisplay(confusion_matrix=cm)
  cm_disp.plot()
  plt.show()

### Regressão Logística

Vamos ver como a regressão logística lida com o problema de análise de sentimentos.

#### Texto vetorizado com Bag of Words.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Instanciando a reg. logistica
reglog = LogisticRegression(max_iter=1000) # Aumentar max_iter para convergência

# Aplicando o modelo
reglog.fit(X1_train, y1_train)

# Predição
y1_reglog_pred = reglog.predict(X1_test)

Analisando as métricas para o modelo com Bag of Words.

In [ ]:
print('Métricas para o modelo com Bag of Words:')
mostra_metricas(y1_test, y1_reglog_pred)

#### Texto vetorizado com TF-IDF.

In [ ]:
reglog2 = LogisticRegression(max_iter=1000) # Aumentar max_iter para convergência

reglog2.fit(X2_train, y2_train)

y2_reglog_pred = reglog2.predict(X2_test)

Analisando as métricas para o modelo com TF-IDF.

In [ ]:
print('Métricas para o modelo com TF-IDF:')
mostra_metricas(y2_test, y2_reglog_pred)

A diferença do desempenho do modelo com os 2 métodos de feature extraction foi pequena, neste caso, mas as métricas apontam a melhor foi a TF-IDF.

### Resultados

Para terminar, vamos avaliar como o modelo atua em novos textos, simulando novas avaliações a partir do modelo que performou melhor, o TF-IDF.

In [ ]:
def nova_predicao(texto):
  # Pré-processa o texto de entrada usando a mesma função de limpeza
  texto_limpo = limpa_texto(texto)

  # Vetoriza o texto limpo usando o TfidfVectorizer treinado
  texto_vetorizado = tfidf_vect.transform([texto_limpo])
  pred = reglog2.predict(texto_vetorizado)

  if pred == 0:
    print(f"'{texto}' -> Essa é uma review negativa.")
  else:
    print(f"'{texto}' -> Essa é uma review positiva.")

In [ ]:
nova_predicao("Demorou muito não gostei")

In [ ]:
nova_predicao("Achei cheirosinho")

In [ ]:
nova_predicao("Nossa que produto ruim é esse parece que encontrei no lixo")

In [ ]:
nova_predicao("Gostei")

In [ ]:
nova_predicao("Não gostei")

In [ ]:
nova_predicao("Que produtinho, hein")

In [ ]:
nova_predicao("apesar do atendimento péssimo, gostei muito do produto!")

In [ ]:
nova_predicao("apesar do atendimento ruim, gostei muito do produto!")

In [ ]:
nova_predicao("apesar do atendimento mediano, gostei muito do produto!")